In [1]:
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

ts = ts.sort_values(["RecordID","Parameter","minutes"]).reset_index(drop=True)

def landmark_table(T):
    d = ts[ts.minutes <= T*60]
    g = d.groupby(["RecordID","Parameter"])
    a = g.agg(last=("Value","last"), mean=("Value","mean"), mn=("Value","min"),
              mx=("Value","max"), n=("Value","size"), t_last=("minutes","max"))
    a["stale_h"] = T - a.t_last/60                      # hours since last measured
    w = (d[d.minutes > (T-6)*60].groupby(["RecordID","Parameter"])
           .agg(w_first=("Value","first"), w_last=("Value","last"),
                t0=("minutes","min"), t1=("minutes","max"), w_mean=("Value","mean")))
    w["slope6"] = (w.w_last - w.w_first) / ((w.t1 - w.t0)/60).where(w.t1 > w.t0)
    f = a.drop(columns="t_last").join(w[["slope6","w_mean"]]).unstack("Parameter")
    f.columns = [f"{p}_{s}" for s, p in f.columns]
    f = f.reset_index()

    cohort = df_out.loc[df_out.Length_of_stay >= T/24,
                        ["RecordID","set","died","SAPS-I","Age","Gender","ICUType","Weight"]]
    tbl = cohort.merge(f, on="RecordID", how="left")
    n_cols = [c for c in tbl.columns if c.endswith("_n")]
    tbl[n_cols] = tbl[n_cols].fillna(0)
    tbl["SAPS-I"] = tbl["SAPS-I"].replace(-1, np.nan)
    return tbl

tables = {T: landmark_table(T) for T in [12, 24, 36]}

def metrics(y, p):
    return dict(AUROC=roc_auc_score(y,p), AUPRC=average_precision_score(y,p), Brier=brier_score_loss(y,p))

meta = ["RecordID","set","died","SAPS-I"]
rows = []
for T, tbl in tables.items():
    tr, te = tbl[tbl.set.isin(["a","b"])], tbl[tbl.set == "c"]
    full = [c for c in tbl.columns if c not in meta]
    nostale = [c for c in full if not c.endswith("_stale_h")]

    lr = make_pipeline(SimpleImputer(), StandardScaler(), LogisticRegression(max_iter=1000))
    lr.fit(tr[["SAPS-I"]], tr.died)
    rows.append(dict(T=T, model="SAPS-I logreg", n_test=len(te), prev=te.died.mean(),
                     **metrics(te.died, lr.predict_proba(te[["SAPS-I"]])[:,1])))

    for name, cols in [("LGBM without staleness", nostale), ("LGBM with staleness", full)]:
        m = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.03, num_leaves=15,
                               subsample=0.8, subsample_freq=1, colsample_bytree=0.6,
                               random_state=0, verbose=-1)
        m.fit(tr[cols], tr.died)
        rows.append(dict(T=T, model=name, n_test=len(te), prev=te.died.mean(),
                         **metrics(te.died, m.predict_proba(te[cols])[:,1])))

results = pd.DataFrame(rows).round(3)
display(results)

NameError: name 'ts' is not defined